# spaCy で大量の文書を高速に処理する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/13_parsing/spacy_pipe.ipynb)

NLP クックブック 13-19 ／ 構文解析

数万件のテキストを spaCy で解析したいが、1 件ずつ `nlp(text)` すると遅い。

In [1]:
!python -m spacy download en_core_web_sm -q

In [2]:
import time
import spacy

texts = ["Apple is looking at buying a U.K. startup for $1 billion.", "The quick brown fox jumps over the lazy dog."] * 1000

def timeit(label, fn):
    t = time.perf_counter()
    result = fn()
    print(f"{label:38} {time.perf_counter() - t:5.1f} 秒")
    return result

nlp = spacy.load("en_core_web_sm")
_ = timeit("1 件ずつ nlp(text)", lambda: [nlp(t) for t in texts])
_ = timeit("nlp.pipe (バッチ処理)", lambda: list(nlp.pipe(texts, batch_size=256)))

nlp_light = spacy.load("en_core_web_sm", exclude=["parser", "ner", "lemmatizer"])  # 品詞だけ欲しい場合
_ = timeit("不要な要素を除外 + pipe", lambda: list(nlp_light.pipe(texts, batch_size=256)))

nlp_tok = spacy.blank("en")  # トークナイズだけ
_ = timeit("トークナイザのみ", lambda: list(nlp_tok.pipe(texts)))

1 件ずつ nlp(text)                          7.6 秒
nlp.pipe (バッチ処理)                         1.9 秒
不要な要素を除外 + pipe                          0.9 秒
トークナイザのみ                                 0.0 秒


In [3]:
# メタデータを一緒に流す (as_tuples) と、ID との対応を保ったまま処理できる
data = [("Tokyo is the capital of Japan.", {"id": 1}), ("Paris is in France.", {"id": 2})]
for doc, ctx in nlp.pipe(data, as_tuples=True):
    print(ctx["id"], [(e.text, e.label_) for e in doc.ents])

1 [('Tokyo', 'GPE'), ('Japan', 'GPE')]
2 [('Paris', 'GPE'), ('France', 'GPE')]


- `nlp.pipe(texts, n_process=2)` でマルチプロセス化もできる (Transformer モデル + GPU の場合は使わない)。
- 文分割だけ必要なら `nlp.add_pipe("sentencizer")` を入れた `spacy.blank` が最速。